**Test with Iris dataset and Test, Validation and Train split function**

In [ ]:
# Aluno: Nicholas Barbosa e Costa
# Matrícula: 95667

from sklearn.datasets import load_iris # carrega o dataset Iris
from sklearn.model_selection import train_test_split # divide o dataset em treino e teste
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay   # matriz de confusão para avaliar o modelo

import numpy as np

import torch
import torch.nn as nn

from torchsummary import summary # resumo do modelo
from torch.utils.tensorboard import SummaryWriter # visualização de métricas no TensorBoard

from datetime import datetime

import matplotlib.pyplot as plt
import copy
from tqdm import tqdm

SEED = 95667

np.random.seed(SEED)
torch.manual_seed(SEED)

# Se estiver usando GPU Apple (MPS):
torch.mps.manual_seed(SEED)

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

X_train_val,X_test,y_train_val,y_test=train_test_split(X,y,test_size=0.15,random_state=SEED)

X_train,X_val,y_train,y_val=train_test_split(X_train_val,y_train_val,test_size=0.2,random_state=SEED)

print(f'Treino: Dados {X_train.shape} Rótulos {y_train.shape}')
print(f'Validação: {X_val.shape} Rótulos {y_val.shape}')
print(f'Teste: {X_test.shape} Rótulos {y_test.shape}')

print(f'Exemplo de dado de treino:    Dados {X_train[1]} Rótulo {y_train[1]}')
print(f'Exemplo de dado de validação: Dados {X_val[1]} Rótulo {y_val[1]}')
print(f'Exemplo de dado de teste:     Dados {X_test[1]} Rótulo {y_test[1]}')

dataset = {}
dataset['train'] = {'data':X_train, 'label':y_train}
dataset['val']   = {'data':X_val, 'label':y_val}
dataset['test']  = {'data':X_test, 'label':y_test}
dataset['class_labels'] = [iris.target_names[0], iris.target_names[1], iris.target_names[2]]

### Architecture definition

In [ ]:
#define ANN architecture as a Torch NN Module
class ANN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.lin1 = nn.Linear(4, 10)
        self.act1 = nn.ReLU()
        self.lin2 = nn.Linear(10, 20)
        self.act2 = nn.ReLU()
        self.lin3 = nn.Linear(20, num_classes)
        
    def forward(self, x, debug=False):
        if debug : print(f'Shape de entrada: {x.shape}')
        x = self.lin1(x)
        if debug : print(f'Shape de entrada: {x.shape}')
        x = self.act1(x)
        if debug : print(f'Shape de entrada: {x.shape}')
        x = self.lin2(x)
        if debug : print(f'Shape de entrada: {x.shape}')
        x = self.act2(x)
        if debug : print(f'Shape de entrada: {x.shape}')
        y = self.lin3(x)
        if debug : print(f'Shape de entrada: {y.shape}')
        return y         

### Model analysis

In [ ]:
if torch.cuda.is_available():
    my_device = torch.device("cuda:0")
else:
    my_device = torch.device("cpu")

print(f"Running on {my_device.type}.")

net = ANN( num_classes=3 )
#net = ANN()

net = net.to(my_device)

a = torch.rand( (1, 4) )
a = a.to(my_device)
b = net( a , debug=True)

del a, b, net

In [ ]:
net = ANN( num_classes=3 )

net = net.to(my_device)

summary(net, input_size=(1, 4), batch_size=1)

del net

### Training functions

In [ ]:
def train ( dataset, prefix=None, upper_bound=101.0, save=False, epochs=100, 
           lr=1e-1, device='cpu', debug=False, layers2tensorboard=True, lambda_reg=0 ) :
    
    num_classes = 3
    
    tensorboard_path = # INSIRA SEU CAMINHO AQUI.
  
    net = ANN( num_classes )
    net.to(device)

    optimizer = torch.optim.SGD(net.parameters(), lr=lr, weight_decay=lambda_reg)
    criterion = nn.CrossEntropyLoss()

    now = datetime.now()
    suffix = now.strftime("%Y%m%d_%H%M%S")
    prefix = suffix if prefix is None else prefix + '-' + suffix  

    writer = SummaryWriter( log_dir=tensorboard_path+prefix )
    
    accuracies = []
    max_accuracy = -1.0  
    
    train_x = torch.from_numpy(dataset['train']['data']).float()
    train_x = train_x.to(device)
    train_label = torch.from_numpy(dataset['train']['label']).float()
    train_label = train_label.to(device)
    
    writer.add_graph(net, train_x)
    
    for epoch in tqdm( range(epochs) , desc='Training epochs...' ) :
        
        # Set Pytorch variables
        net.train()
        optimizer.zero_grad()
                            
        # Forward step
        predict_y = net( train_x )
        
        # Loss
        error = criterion( predict_y , train_label.long() )

        # Back propagation
        error.backward()
        optimizer.step()

        # Accuracies:
        predict_ys = torch.max( predict_y, axis=1 )[1]
        correct    = torch.sum( predict_ys == train_label )
        accuracy_train = correct/train_x.size(0)*100
        
        accuracy_val, error_val = validate(net, dataset['val'], device=device, criterion=criterion)
        accuracies.append(accuracy_val)
        
        # Tensor board writing
        writer.add_scalar( 'Loss/train', error.item(), epoch )
        writer.add_scalar( 'Loss/val', error_val.item(), epoch )
        writer.add_scalar( 'Accuracy/train', accuracy_train, epoch )
        writer.add_scalar( 'Accuracy/val', accuracy_val, epoch )

        # writer.add_scalars('Loss', {
        #     'train': error.item(),
        #     'val': error_val.item()
        # }, epoch)

        # writer.add_scalars('Accuracy', {
        #     'train': accuracy_train,
        #     'val': accuracy_val
        # }, epoch)

        
        if layers2tensorboard :
            plot_layers( net, writer, epoch )

        # Val model
        if accuracy_val > max_accuracy:
            best_model = copy.deepcopy(net)
            max_accuracy = accuracy_val
            print(f'Saving the best model at epoch {epoch+1:3d} ' + 
                    f'with Accuracy: {accuracy_val:8.4f}%')
      
        if debug : print( f'Epoch: {epoch+1:3d} |' 
                         + f'Accuracy Val: {accuracy_val:3.4f}%' )

        if accuracy_val > upper_bound :
            break
   
    if save : 
        models_path = # INSIRA SEU CAMINHO AQUI.
        path = f'{models_path}{prefix}-{max_accuracy:.2f}.pkl'
        torch.save( best_model, path )
        print( f'Model saved in: {path}' )
  
    plt.figure(figsize=(16, 8))
    plt.plot(accuracies)

    writer.flush()
    writer.close()

    return best_model

In [ ]:
def validate ( model , dataset , device='cpu', criterion=None, confusion_matrix_labels=None) :

    x = torch.from_numpy(dataset['data']).float()
    x = x.to(device)
    label = torch.from_numpy(dataset['label']).float()
    label = label.to(device)

    model.eval()

    predict_y = model( x ).detach()
    predict_ys = torch.max( predict_y, axis=1 )[1]
    correct = torch.sum(predict_ys == label)

    accuracy = correct.to('cpu').numpy()*100./x.size(0)

    if confusion_matrix_labels != None :
        disp = ConfusionMatrixDisplay(
            confusion_matrix= confusion_matrix(label.cpu(), predict_ys.cpu()), 
            display_labels=confusion_matrix_labels
        )
        disp.plot(cmap='Blues')
        plt.show()
    
    if criterion == None:
        return accuracy

    error = criterion( predict_y , label.long() ) 

    return accuracy, error

In [ ]:
def plot_layers ( net , writer, epoch ) :
    layers = list(net.modules())

    layer_id = 1
    for layer in layers:
        if isinstance(layer, nn.Linear) :
            writer.add_histogram(f'Bias/linear-{layer_id}', layer.bias, epoch )
            writer.add_histogram(f'Weight/linear-{layer_id}', layer.weight, epoch )
            writer.add_histogram(f'Grad/linear-{layer_id}', layer.weight.grad, epoch )
            layer_id += 1

### Run the training phase

In [ ]:
if torch.cuda.is_available():
    my_device = torch.device("cuda:0")
else:
    my_device = torch.device("cpu")
    
print(f"Running on {my_device.type}")
    
dataset_name = 'Iris'
epochs = 500
lr = 5e-1
lambda_reg = 1e-5
prefix = 'ANN-{}-e-{}-lr-{}'.format(dataset_name, epochs, lr)

net = train( dataset=dataset, epochs=epochs, device=my_device, 
            upper_bound=101.0, lr=lr, lambda_reg=lambda_reg,
            layers2tensorboard=True, save=False, prefix=prefix
           )

## Executar o teste

In [ ]:
accuracy_test = validate ( net , 
                          dataset['test'] , 
                          device=my_device, 
                          confusion_matrix_labels=dataset['class_labels']
                         )
print(f"A acurácia do modelo treinado no conjunto de teste é: {accuracy_test:.2f}% ")

## Teste de inferência única

In [ ]:
print(net)

test_x = torch.from_numpy(dataset['test']['data']).float()
test_x = test_x.to(my_device)
test_label = torch.from_numpy(dataset['test']['label']).float()
test_label = test_label.to(my_device)

indice  = np.random.randint(0,dataset['test']['label'].shape[0])

print(f'Entrada do modelo: {test_x[indice].detach().cpu().numpy()}')
output = net(test_x[indice])
print(f'Saída raw do modelo: {output.detach().cpu().numpy()}')
predicted_class = torch.max(output, dim=0)[1]

predicted_class = predicted_class.to('cpu').numpy()

predicted_class_name = dataset['class_labels'][predicted_class]
correct_name = dataset['class_labels'][y_test[indice]]
print("\nHIT" if predicted_class_name == correct_name else "MISS")
print(f"Saída processada do modelo foi {predicted_class_name} e saída correta era {correct_name} para a amostra indice {indice}")